In [ ]:
!pip install transformers bitsandbytes peft accelerate datasets trl PyMuPDF -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 62.4 MB/s eta 0:00:00


In [ ]:
import fitz
from datasets import Dataset

In [ ]:
def extract_text_from_pdf(path):
  text_blocks = []
  with fitz.open(path) as pdf:
    for page in pdf:
      text = page.get_text("text").strip()
      if text:
        text_blocks.append(text)

  return text_blocks

In [ ]:
texts = extract_text_from_pdf("/content/M6.pdf")

In [ ]:
import re

def split_paragraphs(pages):
  paragraphs = []
  for page_text in pages:
    chunks = re.split(r"\n\s*\n", page_text)
    for chunk in chunks:
      clean = chunk.strip()
      if len(clean) > 30:
        paragraphs.append(clean)

  return paragraphs

In [ ]:
chunks = split_paragraphs(texts)

In [ ]:
len(chunks)

131

In [ ]:
data = [{'text': c} for c in chunks]

In [ ]:
dataset = Dataset.from_list(data)

In [ ]:
dataset

Dataset({
    features: ['text'],
    num_rows: 131
})

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [ ]:
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize_fn(examples):
  tokens = tokenizer(examples['text'], truncation=True, padding='max_length', max_length=512)
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

In [ ]:
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=['text'])

Map:   0%|          | 0/131 [00:00<?, ? examples/s]

In [ ]:
tokenized

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 131
})

In [ ]:
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
lora_config = LoraConfig(
    task_type = TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [ ]:
q_lora_model = get_peft_model(model, lora_config)

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/trained_model",
    per_device_train_batch_size=2,
    num_train_epochs=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    report_to="none")

In [ ]:
trainer = Trainer(
    model = q_lora_model,
    args = training_args,
    train_dataset = tokenized
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
50,11.978038
100,9.460286


TrainOutput(global_step=132, training_loss=10.249556801535867, metrics={'train_runtime': 226.0301, 'train_samples_per_second': 1.159, 'train_steps_per_second': 0.584, 'total_flos': 833548374245376.0, 'train_loss': 10.249556801535867, 'epoch': 2.0})

In [ ]:
model_path = "/content/trained_model/checkpoint-132"

In [ ]:
trained_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [ ]:
prompt = "How to reach the non-target customers"

In [ ]:
input = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = trained_model.generate(
    **input,
    max_new_tokens=100,
    do_sample=True,
    top_p=0.9,
    temperature=0.8,
    repetition_penalty=1.1
)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


In [ ]:
tokenizer.decode(outputs[0], skip_special_tokens=True)

'How to reach the non-target customers in an email marketing campaign?'

In [ ]:
import shutil
from google.colab import files

# Zip the folder
shutil.make_archive('trained_model', 'zip', '/content/trained_model')

# Download the zip
files.download('trained_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>